In [1]:
import pandas as pd
import numpy as np
import duckdb
import polars as pl

In [4]:
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/ML_Dataset/open_food_facts_2025 (1).csv'

df_preview = pd.read_csv(file_path, nrows=5)

display(df_preview)

,code,url,product_name,generic_name,quantity,countries,countries_en,origins,manufacturing_places,categories,...,sodium_100g,completeness,unique_scans_n,serving_size,creator,image_url,main_category_en,nova_group_label,created_date,last_modified_date
0,7,http://world-en.openfoodfacts.org/product/0000...,granola Bio le Chocolaté,NaN,450g,"Spanien, Germany","Germany,Spain",NaN,NaN,"Aliments et boissons à base de végétaux, Alime...",...,NaN,0.7625,2.0,25 gram,smoothie-app,https://images.openfoodfacts.org/images/produc...,Dried fruits,NaN,2023-03-14,2025-05-26
1,9,http://world-en.openfoodfacts.org/product/0000...,xytitol pastilles,NaN,700ml,"Germany,Spain","Germany,Spain",NaN,NaN,"it:Gestione sovrappeso, it:obesità, Wrap",...,NaN,0.5750,1.0,0g,openfoodfacts-contributors,https://images.openfoodfacts.org/images/produc...,it:obesita,NaN,2018-05-25,2025-06-11
2,13,http://world-en.openfoodfacts.org/product/0000...,Powdered peanut butter,NaN,300 g,en:Switzerland,Switzerland,NaN,NaN,"Snacks, Meals, Rice dishes, Risottos, Powder p...",...,NaN,0.9750,2.0,0g,jeff-krab,https://images.openfoodfacts.org/images/produc...,Powder-peanut-butter,NaN,2019-03-30,2025-04-10
3,15,http://world-en.openfoodfacts.org/product/0000...,Madeleines ChocoLait,NaN,1000 g,France,France,NaN,NaN,"Snacks, Snacks sucrés, Biscuits et gâteaux, Gâ...",...,NaN,0.7875,1.0,"21,6g",openfoodfacts-contributors,https://images.openfoodfacts.org/images/produc...,Chocolate madeleines,Ultra-processed foods,2018-04-15,2025-05-06
4,22,http://world-en.openfoodfacts.org/product/0000...,Farandole de madeleine,NaN,590 g,Brasil,Brazil,NaN,NaN,"Snacks, Snacks sucrés, Biscuits et gâteaux, Gâ...",...,NaN,0.8000,1.0,54g,openfoodfacts-contributors,https://images.openfoodfacts.org/images/produc...,Long madeleines,Ultra-processed foods,2021-02-28,2025-05-06


In [5]:
columns = [
    'categories_en',
    'url',
    'main_category_en',
    'labels',
    'labels_en',
    'unique_scans_n',
    'energy-kcal_100g',
    'fat_100g',
    'saturated-fat_100g',
    'carbohydrates_100g',
    'sugars_100g'
]

df = pd.read_csv(
    file_path,
    usecols=columns,
    low_memory=False
)

print("Dataset shape:", df.shape)
display(df.head(20))

Dataset shape: (1350091, 11)


,url,categories_en,labels,labels_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,unique_scans_n,main_category_en
0,http://world-en.openfoodfacts.org/product/0000...,"Plant-based foods and beverages,Plant-based fo...",en:made-in-france,Made in France,NaN,NaN,NaN,NaN,NaN,2.0,Dried fruits
1,http://world-en.openfoodfacts.org/product/0000...,"Sandwiches,Wraps,it:gestione-sovrappeso,it:obe...","Organic, Vegetarian, EU Organic, Vegan, DE-ÖKO...","Organic,Vegetarian,EU Organic,Vegan,DE-ÖKO-006...",NaN,NaN,NaN,NaN,NaN,1.0,it:obesita
2,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Meals,Rice dishes,Risottos,Powder-peanu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Powder-peanut-butter
3,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,C...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Chocolate madeleines
4,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,C...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Long madeleines
5,http://world-en.openfoodfacts.org/product/0000...,"Sandwiches,Wraps","No gluten, Vegetarian, Vegan, en:usda-organic","No gluten,Organic,Vegetarian,USDA Organic,Vegan",NaN,NaN,NaN,NaN,NaN,1.0,Wraps
6,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,M...","Vegetarian, Vegan, en:no-additives","Vegetarian,Vegan,No additives",NaN,NaN,NaN,NaN,NaN,1.0,Long madeleines
7,http://world-en.openfoodfacts.org/product/0000...,"Plant-based foods and beverages,Plant-based fo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Pastas
8,http://world-en.openfoodfacts.org/product/0000...,pt:pao-com-queijo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,pt:pao-com-queijo
9,http://world-en.openfoodfacts.org/product/0000...,"Plant-based foods and beverages,Plant-based fo...","Point Vert, Fabriqué en France","Green Dot,Made in France",NaN,NaN,NaN,NaN,NaN,12.0,Jams


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1350091 entries, 0 to 1350090
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   url                 1350091 non-null  object 
 1   categories_en       1349999 non-null  object 
 2   labels              566939 non-null   object 
 3   labels_en           566941 non-null   object 
 4   energy-kcal_100g    928156 non-null   float64
 5   fat_100g            928105 non-null   float64
 6   saturated-fat_100g  924577 non-null   float64
 7   carbohydrates_100g  927270 non-null   float64
 8   sugars_100g         926484 non-null   float64
 9   unique_scans_n      732768 non-null   float64
 10  main_category_en    1349999 non-null  object 
dtypes: float64(6), object(5)
memory usage: 113.3+ MB


In [9]:
df['labels'].value_counts()


,count
labels,
Point Vert,15436
Organic,11455
No gluten,9515
Punto Verde,8059
"No GMOs, Non GMO project",7170
...,...
"Halal, fr:Association Rituelle de la Grande Mosquée de Lyon, en:french-beef",1
"Viande Française, Sans conservateurs, Viande Bovine Française, Halal",1
"en:French meat, en:French poultry, Halal",1


In [10]:
df['labels_en'].value_counts()

,count
labels_en,
Green Dot,32191
No gluten,30586
Organic,19044
"Vegetarian,Vegan",13672
No preservatives,8657
...,...
"Low or no sugar,Low sugar,Reduced sugar,55% less sugar,No added sugar",1
Vegitarisch,1
"Vegetarian,Sustainable farming,Vegan,No palm oil,The Vegan Society,UTZ Certified,UTZ Certified Cocoa",1


Notes:-
`1. Imputing missing values in labels_en with values in label is very difficult because i have to translate it.

2. Since both things are same only difference is the language I'm dropping labels

In [23]:
df = df.drop(columns=["labels"])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1350091 entries, 0 to 1350090
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   url                 1350091 non-null  object 
 1   categories_en       1349999 non-null  object 
 2   labels              566939 non-null   object 
 3   labels_en           566941 non-null   object 
 4   energy-kcal_100g    928156 non-null   float64
 5   fat_100g            928105 non-null   float64
 6   saturated-fat_100g  924577 non-null   float64
 7   carbohydrates_100g  927270 non-null   float64
 8   sugars_100g         926484 non-null   float64
 9   unique_scans_n      732768 non-null   float64
 10  main_category_en    1349999 non-null  object 
dtypes: float64(6), object(5)
memory usage: 113.3+ MB


In [12]:

# A serving of 100g cannot have an ingredient more than 100g
df = df[
    (df["fat_100g"].isna() | df["fat_100g"].between(0, 100))
    & (df["saturated-fat_100g"].isna() | df["saturated-fat_100g"].between(0, 100))
    & (df["carbohydrates_100g"].isna() | df["carbohydrates_100g"].between(0, 100))
    & (df["sugars_100g"].isna() | df["sugars_100g"].between(0, 100))
]


In [13]:
df.head()

,url,categories_en,labels,labels_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,unique_scans_n,main_category_en
0,http://world-en.openfoodfacts.org/product/0000...,"Plant-based foods and beverages,Plant-based fo...",en:made-in-france,Made in France,NaN,NaN,NaN,NaN,NaN,2.0,Dried fruits
1,http://world-en.openfoodfacts.org/product/0000...,"Sandwiches,Wraps,it:gestione-sovrappeso,it:obe...","Organic, Vegetarian, EU Organic, Vegan, DE-ÖKO...","Organic,Vegetarian,EU Organic,Vegan,DE-ÖKO-006...",NaN,NaN,NaN,NaN,NaN,1.0,it:obesita
2,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Meals,Rice dishes,Risottos,Powder-peanu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Powder-peanut-butter
3,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,C...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Chocolate madeleines
4,http://world-en.openfoodfacts.org/product/0000...,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,C...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Long madeleines


In [14]:
df["categories_en"] = (
    df["categories_en"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.split(",")
    .str[0]
    .str.strip()
)

In [15]:
df['categories_en'].isna().sum()

np.int64(0)

In [16]:
nutrient_cols = ["fat_100g", "saturated-fat_100g", "sugars_100g", "carbohydrates_100g"]

for col in nutrient_cols:
  # 1. Fill missing values with the median of their specific category
  df[col] = df[col].fillna(df.groupby("categories_en")[col].transform("median"))

  # 2. Fallback: If an entire category is missing/empty, fill with the global median
  df[col] = df[col].fillna(df[col].median())

In [17]:
df.isna().sum()

,0
url,0
categories_en,0
labels,782846
labels_en,782844
energy-kcal_100g,421935
fat_100g,0
saturated-fat_100g,0
carbohydrates_100g,0
sugars_100g,0
unique_scans_n,617157


For missing numerical nutrition data, you can impute missing values using macronutrient formulas ($1\text{g Fat} \approx 9\text{ kcal}$, $1\text{g Carbs} \approx 4\text{ kcal}$, $1\text{g Protein} \approx 4\text{ kcal}$) since fat_100g, carbohydrates_100g, etc., have 0 missing values in your dataset.

In [18]:
# Formula-based calculation: (Fat * 9) + (Carbs * 4) + (Protein * 4 if available)
calculated_kcal = (df['fat_100g'] * 9) + (df['carbohydrates_100g'] * 4)

# Fill missing energy-kcal_100g using calculated values
df['energy-kcal_100g'] = df['energy-kcal_100g'].fillna(calculated_kcal)

# Handle extreme outliers (e.g., negative values or values > 1000 kcal per 100g)
df['energy-kcal_100g'] = df['energy-kcal_100g'].clip(lower=0, upper=900)

This high-missingness text feature contains organic/fair-trade/quality labels. Missing values generally mean the product has no special labels.

In [19]:
# Fill missing text with 'None' or empty string
df['labels_en'] = df['labels_en'].fillna('None')

# Convert string tags into a clean list format or binary flags
df['has_organic_label'] = df['labels_en'].str.contains('organic', case=False, na=False).astype(int)

This is a count/popularity metric. A missing value indicates the product was created or imported without scan metadata, representing 0 scans.

In [20]:
# Impute missing scan counts with 0
df['unique_scans_n'] = df['unique_scans_n'].fillna(0)

In [21]:
df['main_category_en'] = df['main_category_en'].fillna('Unknown')

In [24]:
df.isna().sum()

,0
url,0
categories_en,0
labels_en,0
energy-kcal_100g,0
fat_100g,0
saturated-fat_100g,0
carbohydrates_100g,0
sugars_100g,0
unique_scans_n,0
main_category_en,0


In [25]:
df[df.duplicated()]

,url,categories_en,labels_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,unique_scans_n,main_category_en,has_organic_label
746851,http://world-en.openfoodfacts.org/product/3760...,beverages and beverages preparations,None,26.000000,0.000000,0.0,6.50,5.30,0.0,Apple juices,0
746886,http://world-en.openfoodfacts.org/product/3760...,condiments,"Organic,EU Organic,AB Agriculture Biologique",98.700000,7.700000,1.0,7.35,4.20,0.0,Barbecue sauces,1
746890,http://world-en.openfoodfacts.org/product/3760...,plant-based foods and beverages,"Organic,EU Organic,AB Agriculture Biologique",117.585012,2.576112,0.5,23.60,3.04,0.0,fr:muesli-chocolat-noisette,1
746919,http://world-en.openfoodfacts.org/product/3760...,snacks,None,413.000000,21.000000,8.4,56.00,30.00,0.0,fr:cookies-pepites-de-chocolat,0
1008938,http://world-en.openfoodfacts.org/product/6194...,plant-based foods and beverages,"Halal,fr:brc,fr:ifs",117.585012,2.576112,0.5,23.60,3.04,0.0,Tomatoes,0


In [26]:
df.to_parquet('sahil_output.parquet')